# Predicting Late Deliveries and Understanding the Key Drivers of Customer Satisfaction in a Food-Delivery Platform

**AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026**  
**Conducted by BharatCares in association with AICTE and IBM SkillsBuild**

---

## Section 1: Project Introduction

This project is submitted as part of the **AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026**, conducted by BharatCares in association with AICTE and IBM SkillsBuild.

The dataset used is a **cleaned, structured food-delivery transaction dataset** (`Swiggy Cleaned Data by MKP.xlsx`), containing **10,000 order records** across **12 Indian cities**, **20 restaurants**, and **8 cuisine categories**, spanning April 2024 to March 2026.

> **Important Note on Data Origin:**  
> This dataset is a pre-cleaned educational dataset used for analytical practice. It is **not** proprietary Swiggy corporate data, and **no claims** are made about Swiggy's actual operations. The dataset simulates realistic food-delivery transaction patterns for the purpose of learning data analytics techniques.

The workbook was received in **already-cleaned form**. The cleaning steps are documented in two companion sheets within the workbook (`How I Cleaned the Data` and `Cleaning Summary`) and are reproduced in Section 7 of this notebook. The original workbook has **not been modified** in any way.

**IBM Bob** was used as the AI-assisted development and analysis component throughout this project. Its role is documented in Section 16.

---

## Section 2: Business Problem

A food-delivery platform needs to:

1. **Understand operational factors associated with delivery delays** — which cities, restaurants, time periods, and order characteristics are linked to late deliveries.
2. **Understand what drives customer satisfaction** — how delivery timing, order characteristics, and other factors relate to customer ratings.
3. **Build a predictive model** that identifies orders likely to arrive late, enabling proactive operational intervention.

**Why this matters:**  
Delivery speed is strongly associated with customer satisfaction in food delivery (in this dataset, delivery delay shows a moderate negative correlation with customer ratings). Identifying late-delivery risk before or at dispatch allows the platform to re-route, notify customers proactively, or prioritise logistics resources.

---

## Section 3: Objectives

1. Understand the Swiggy-style food-delivery dataset structure and contents.
2. Validate data quality and document the existing cleaning work.
3. Analyze delivery performance across cities, restaurants, time periods, customer segments, and order characteristics.
4. Engineer key features including `delivery_delay`, `is_late`, temporal features, and customer/order bands.
5. Analyze the relationship between delivery delay and customer rating.
6. Identify factors associated with customer satisfaction.
7. Build and compare classification models for late-delivery prediction.
8. Evaluate models using appropriate classification metrics.
9. Interpret model results and understand feature importance.
10. Analyze order cancellation and refund patterns.
11. Convert verified findings into practical business recommendations.
12. Demonstrate the role of IBM Bob as the AI-assisted analysis component.

---

## Section 4: Dataset Overview

| Attribute | Value |
|---|---|
| File | `Swiggy Cleaned Data by MKP.xlsx` |
| Sheet used | `Cleaned Data` |
| Total records | 10,000 |
| Total columns | 28 |
| Date range | April 2024 – March 2026 |
| Cities | 12 |
| Restaurants | 20 |
| Cuisine categories | 8 |
| Customer segments | 5 |
| Payment modes | 5 |

### Column Reference

| Column | Description | Type |
|---|---|---|
| Order ID | Unique order identifier | String |
| Order Date | Date of order (YYYY-MM-DD) | String/Date |
| Order Time | Time of order (HH:MM) | String |
| Customer ID | Anonymised customer identifier | String |
| Customer Name | Anonymised (Customer ####) | String |
| Customer Age | Age in years (18–60) | Integer |
| Customer Gender | Male / Female / Other | Categorical |
| Customer Segment | Young Professional / Working Professional / Student / Family / Premium | Categorical |
| City | 12 Indian cities | Categorical |
| State | 11 states | Categorical |
| Region | North / South / East / West / Central | Categorical |
| Restaurant ID | Unique restaurant code | String |
| Restaurant Name | Restaurant name | String |
| Restaurant Category | Cuisine type (8 categories) | Categorical |
| Food Type | Vegetarian / Non-Vegetarian | Categorical |
| Sub Category | Item sub-type (22 categories) | Categorical |
| Item Count | Number of items ordered (1–8) | Integer |
| Order Value INR | Gross order value in ₹ | Float |
| Discount Pct | Discount percentage applied | Float |
| Discount Amount INR | Discount value in ₹ | Float |
| Delivery Fee INR | Delivery charge in ₹ | Float |
| Final Amount INR | Net amount paid by customer | Float |
| Delivery Distance KM | Distance from restaurant to customer | Float |
| Estimated Delivery Min | Platform estimate of delivery time | Integer |
| Actual Delivery Min | Actual time taken for delivery | Integer |
| Payment Mode | UPI / Credit Card / Debit Card / Wallet / Cash on Delivery | Categorical |
| Customer Rating | Customer rating (2.5–5.0) | Float |
| Order Status | Delivered / Cancelled / Refunded | Categorical |

---

## Section 5: Data Loading

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, kruskal, chi2_contingency

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

# ── Plot styling ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
SEED = 42

print('Libraries loaded successfully.')

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
FILE = 'Swiggy Cleaned Data by MKP.xlsx'
SHEET = 'Cleaned Data'

df_raw = pd.read_excel(FILE, sheet_name=SHEET)

print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head(3)

In [ ]:
# ── Preserve original; work on a copy ────────────────────────────────────────
df = df_raw.copy()

# Standardise column names: strip whitespace
df.columns = df.columns.str.strip()

# Parse Order Date
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')

# Parse Order Time to extract hour
df['Order Time'] = df['Order Time'].astype(str).str.strip()

print('Date range:', df['Order Date'].min().date(), 'to', df['Order Date'].max().date())
print('Null dates after parse:', df['Order Date'].isna().sum())
df.dtypes

---

## Section 6: Data Quality Validation

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

print('=== Missing Values ===')
print(missing_df.to_string() if not missing_df.empty else 'No missing values detected.')

In [ ]:
# ── Duplicate check ───────────────────────────────────────────────────────────
dup_count = df.duplicated(subset='Order ID').sum()
print(f'Duplicate Order IDs: {dup_count}')
print(f'Total rows: {len(df)}')
print(f'Unique Order IDs: {df["Order ID"].nunique()}')

In [ ]:
# ── Numeric ranges validation ─────────────────────────────────────────────────
numeric_cols = [
    'Customer Age', 'Item Count', 'Order Value INR', 'Discount Pct',
    'Discount Amount INR', 'Delivery Fee INR', 'Final Amount INR',
    'Delivery Distance KM', 'Estimated Delivery Min',
    'Actual Delivery Min', 'Customer Rating'
]
df[numeric_cols].describe().round(2)

In [ ]:
# ── Categorical value counts ──────────────────────────────────────────────────
cat_cols = [
    'Customer Gender', 'Customer Segment', 'City', 'Region',
    'Restaurant Category', 'Food Type', 'Payment Mode', 'Order Status'
]
for col in cat_cols:
    vc = df[col].value_counts(dropna=False)
    print(f'\n--- {col} ({df[col].nunique()} unique) ---')
    print(vc.to_string())

In [ ]:
# ── Logical consistency check ─────────────────────────────────────────────────
# Actual Delivery Min range
print('Actual Delivery Min — min:', df['Actual Delivery Min'].min(),
      ' max:', df['Actual Delivery Min'].max())
print('Estimated Delivery Min — min:', df['Estimated Delivery Min'].min(),
      ' max:', df['Estimated Delivery Min'].max())

# Check negative delivery fees or order values
print('Negative Order Values:', (df['Order Value INR'] < 0).sum())
print('Negative Delivery Fees:', (df['Delivery Fee INR'] < 0).sum())
print('Customer Age < 18:', (df['Customer Age'] < 18).sum())
print('Customer Age > 60:', (df['Customer Age'] > 60).sum())
print('\nOrder Status counts:')
print(df['Order Status'].value_counts())

---

## Section 7: Existing Cleaning Documentation

The dataset was received in pre-cleaned form. The following cleaning steps were performed by the dataset author and are documented in the companion sheets of the workbook.

### Cleaning Summary (from workbook sheet: *Cleaning Summary*)

| Attribute | Value |
|---|---|
| Raw records | 10,040 |
| Clean records | 10,000 |
| Records removed | 40 duplicate records |
| Purpose | Prepare the dataset for business analysis and visualization |

### Cleaning Steps (from workbook sheet: *How I Cleaned the Data*)

| Step | What Was Checked | What Was Done | Why It Was Important |
|---|---|---|---|
| 1 | Duplicate records | Checked Order ID and removed 40 duplicate transaction rows | Prevents the same order from being counted more than once |
| 2 | Missing Customer ID | Checked 45 blank Customer ID values; kept them missing where the customer could not be reliably identified | Avoids creating false customer information |
| 3 | Missing customer ratings | Checked 75 blank ratings; kept them blank | A missing rating cannot be reliably guessed |
| 4 | Missing city | Checked 35 blank city values; kept them missing where location was unavailable | Avoids inventing geographic information |
| 5 | City formatting | Found inconsistent spaces and capitalisation in 80 city values; removed extra spaces and standardised format | Ensures cities are grouped correctly in analysis |
| 6 | Category formatting | Found 60 inconsistent category labels; standardised to consistent category names | Prevents one category from being split into multiple groups |
| 7 | Payment mode formatting | Standardised 30 inconsistent payment mode labels | Makes payment analysis accurate and consistent |
| 8 | Date formatting | Found 60 dates in mixed formats; converted to one consistent date format | Allows correct time-based analysis and charts |
| 9 | Numeric values | Checked 50 order values stored as text or currency string; converted to numeric INR values | Makes calculations, summaries and visualisations work correctly |
| 10 | Delivery time validation | Checked 20 unrealistic delivery time values; validated using documented cleaning rule | Protects delivery performance measures from incorrect values |
| 11 | Final validation | Rechecked cleaned data for duplicates, missing values, formatting and logical consistency | Confirms the final dataset is ready for analysis |

> **Acknowledgement:** All cleaning work documented above was performed by the dataset author prior to this analysis. This project builds its analysis on top of the already-cleaned dataset without modifying the original workbook.

---

## Section 8: Feature Engineering

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CORE DERIVED FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

# 1. Delivery Delay (continuous)
# Definition: how many minutes late (or early) the delivery was.
# Positive = late; negative = early; zero = exactly on time.
df['delivery_delay'] = df['Actual Delivery Min'] - df['Estimated Delivery Min']

# 2. Is Late (binary ML target)
# 1 = arrived after estimated time; 0 = on time or early.
df['is_late'] = (df['Actual Delivery Min'] > df['Estimated Delivery Min']).astype(int)

print('=== delivery_delay ===')
print(df['delivery_delay'].describe().round(2))
print(f'\nLate orders: {df["is_late"].sum()} ({df["is_late"].mean()*100:.1f}%)')
print(f'On-time/early orders: {(df["is_late"]==0).sum()} ({(df["is_late"]==0).mean()*100:.1f}%)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEMPORAL FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

# order_hour: hour of day (0–23)
df['order_hour'] = df['Order Time'].str.split(':').str[0].astype(float).fillna(-1).astype(int)

# day_of_week: 0=Monday … 6=Sunday
df['day_of_week'] = df['Order Date'].dt.dayofweek

# month: 1–12
df['month'] = df['Order Date'].dt.month

# year
df['year'] = df['Order Date'].dt.year

# weekend indicator: 1 if Saturday or Sunday
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# day_name for display
df['day_name'] = df['Order Date'].dt.day_name()

print('Temporal features created.')
print(df[['order_hour', 'day_of_week', 'month', 'year', 'is_weekend']].describe().round(2))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# BAND FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

# distance_band: categorise delivery distance into operational bands
df['distance_band'] = pd.cut(
    df['Delivery Distance KM'],
    bins=[0, 3, 6, 10, 20],
    labels=['Short (≤3 km)', 'Medium (3–6 km)', 'Long (6–10 km)', 'Very Long (>10 km)']
)

# age_group: standard demographic groupings
df['age_group'] = pd.cut(
    df['Customer Age'],
    bins=[17, 24, 34, 44, 60],
    labels=['18–24', '25–34', '35–44', '45–60']
)

# order_value_band: low/medium/high spend
df['order_value_band'] = pd.cut(
    df['Order Value INR'],
    bins=[0, 300, 600, 1000, 3000],
    labels=['Low (<₹300)', 'Medium (₹300–600)', 'High (₹600–1000)', 'Very High (>₹1000)']
)

# time_of_day: broader time buckets for analysis and modelling
def hour_to_tod(h):
    if 6 <= h < 12: return 'Morning'
    elif 12 <= h < 16: return 'Afternoon'
    elif 16 <= h < 20: return 'Evening'
    elif 20 <= h <= 23: return 'Night'
    else: return 'Late Night'

df['time_of_day'] = df['order_hour'].apply(hour_to_tod)

print('Band features created.')
print('\ndistance_band:')
print(df['distance_band'].value_counts().sort_index())
print('\nage_group:')
print(df['age_group'].value_counts().sort_index())
print('\norder_value_band:')
print(df['order_value_band'].value_counts().sort_index())
print('\ntime_of_day:')
print(df['time_of_day'].value_counts())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ADDITIONAL DERIVED METRICS
# ═══════════════════════════════════════════════════════════════════════════════

# value_per_item: average value per item ordered (order complexity proxy)
df['value_per_item'] = df['Order Value INR'] / df['Item Count']

# delivery_speed_kmpm: km covered per actual minute
df['delivery_speed_kmpm'] = df['Delivery Distance KM'] / df['Actual Delivery Min']

# discount_flag: binary — whether any discount was applied
df['discount_flag'] = (df['Discount Pct'] > 0).astype(int)

print('Additional features created.')
print(df[['value_per_item', 'delivery_speed_kmpm', 'discount_flag']].describe().round(3))

In [ ]:
# ── Final feature engineering summary ────────────────────────────────────────
engineered = [
    ('delivery_delay', 'Actual − Estimated Delivery Min (continuous)', 'EDA + analysis'),
    ('is_late', 'Binary: 1 if late, 0 if on time', 'ML target'),
    ('order_hour', 'Hour of day (0–23)', 'ML + EDA'),
    ('day_of_week', 'Day of week (0=Mon, 6=Sun)', 'ML + EDA'),
    ('month', 'Month (1–12)', 'EDA seasonality'),
    ('is_weekend', 'Binary weekend flag', 'ML + EDA'),
    ('time_of_day', 'Morning/Afternoon/Evening/Night/Late Night', 'EDA + ML'),
    ('distance_band', 'Distance category (4 bands)', 'EDA + ML'),
    ('age_group', 'Customer age group (4 bands)', 'EDA + ML'),
    ('order_value_band', 'Order value category (4 bands)', 'EDA'),
    ('value_per_item', 'Order Value / Item Count', 'ML feature'),
    ('discount_flag', 'Binary: discount applied or not', 'ML feature'),
]
summary_fe = pd.DataFrame(engineered, columns=['Feature', 'Definition', 'Usage'])
print(summary_fe.to_string(index=False))

---

## Section 9: Exploratory Data Analysis (EDA)

In [ ]:
# ── 9.1 Order value distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Order Value INR'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Order Value (INR)')
axes[0].set_xlabel('Order Value (₹)')
axes[0].set_ylabel('Number of Orders')
axes[0].axvline(df['Order Value INR'].mean(), color='orange', linestyle='--',
                label=f'Mean: ₹{df["Order Value INR"].mean():.0f}')
axes[0].axvline(df['Order Value INR'].median(), color='red', linestyle=':',
                label=f'Median: ₹{df["Order Value INR"].median():.0f}')
axes[0].legend()

axes[1].hist(df['Customer Rating'].dropna(), bins=30, color='teal', edgecolor='white')
axes[1].set_title('Distribution of Customer Ratings')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Number of Orders')
axes[1].axvline(df['Customer Rating'].mean(), color='orange', linestyle='--',
                label=f'Mean: {df["Customer Rating"].mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_01_distributions.png', bbox_inches='tight')
plt.show()
print('\nOrder Value INR Stats:')
print(df['Order Value INR'].describe().round(2))

In [ ]:
# ── 9.2 Orders by hour of day ─────────────────────────────────────────────────
hourly = df[df['order_hour'] >= 0].groupby('order_hour').size()

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(hourly.index, hourly.values, color='steelblue', edgecolor='white')
ax.set_title('Order Volume by Hour of Day')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Orders')
ax.set_xticks(range(0, 24))
for bar in bars:
    if bar.get_height() > 500:
        bar.set_color('darkorange')
plt.tight_layout()
plt.savefig('fig_02_hourly_orders.png', bbox_inches='tight')
plt.show()
print('Peak hours (top 5):')
print(hourly.sort_values(ascending=False).head())

In [ ]:
# ── 9.3 Monthly order volume trend ───────────────────────────────────────────
monthly = df.groupby(df['Order Date'].dt.to_period('M')).size()
monthly.index = monthly.index.astype(str)

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(monthly.index, monthly.values, marker='o', color='steelblue', linewidth=2)
ax.set_title('Monthly Order Volume Trend (Apr 2024 – Mar 2026)')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Orders')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('fig_03_monthly_trend.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 9.4 Orders and revenue by city ──────────────────────────────────────────
city_stats = df.groupby('City').agg(
    Order_Count=('Order ID', 'count'),
    Total_Revenue=('Final Amount INR', 'sum'),
    Avg_Order_Value=('Order Value INR', 'mean')
).sort_values('Order_Count', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

city_stats['Order_Count'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Number of Orders by City')
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of Orders')
axes[0].tick_params(axis='x', rotation=45)

(city_stats['Avg_Order_Value']).plot(kind='bar', ax=axes[1], color='teal', edgecolor='white')
axes[1].set_title('Average Order Value (₹) by City')
axes[1].set_xlabel('')
axes[1].set_ylabel('Avg Order Value (₹)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('fig_04_city_stats.png', bbox_inches='tight')
plt.show()
print(city_stats.round(2))

In [ ]:
# ── 9.5 Restaurant category share ────────────────────────────────────────────
cat_vc = df['Restaurant Category'].value_counts()

fig, ax = plt.subplots(figsize=(9, 9))
wedges, texts, autotexts = ax.pie(
    cat_vc.values, labels=cat_vc.index,
    autopct='%1.1f%%', startangle=90,
    colors=sns.color_palette('muted', len(cat_vc))
)
ax.set_title('Order Share by Restaurant Category')
plt.tight_layout()
plt.savefig('fig_05_category_share.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 9.6 Payment mode distribution ────────────────────────────────────────────
pay_vc = df['Payment Mode'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
pay_vc.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Orders by Payment Mode')
ax.set_ylabel('Number of Orders')
ax.tick_params(axis='x', rotation=30)

for i, v in enumerate(pay_vc.values):
    ax.text(i, v + 20, f'{v/len(df)*100:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('fig_06_payment_mode.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 9.7 Customer segment distribution ────────────────────────────────────────
seg_vc = df['Customer Segment'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
seg_vc.plot(kind='bar', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Orders by Customer Segment')
axes[0].set_ylabel('Number of Orders')
axes[0].tick_params(axis='x', rotation=30)

df.groupby('Customer Segment')['Order Value INR'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Average Order Value (₹) by Customer Segment')
axes[1].set_ylabel('Avg Order Value (₹)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('fig_07_segment_stats.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 9.8 Correlation heatmap (numeric features) ───────────────────────────────
corr_cols = [
    'Customer Age', 'Item Count', 'Order Value INR', 'Discount Pct',
    'Delivery Fee INR', 'Final Amount INR', 'Delivery Distance KM',
    'Estimated Delivery Min', 'Actual Delivery Min', 'Customer Rating',
    'delivery_delay', 'order_hour', 'is_late'
]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    ax=ax, annot_kws={'size': 9}
)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('fig_08_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 9.9 Delivery delay distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['delivery_delay'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='On Time (0)')
axes[0].set_title('Distribution of Delivery Delay (minutes)')
axes[0].set_xlabel('Delivery Delay (Actual − Estimated, minutes)')
axes[0].set_ylabel('Number of Orders')
axes[0].legend()

late_counts = df['is_late'].value_counts()
axes[1].bar(['On Time / Early', 'Late'],
            [late_counts.get(0, 0), late_counts.get(1, 0)],
            color=['teal', 'salmon'], edgecolor='white')
axes[1].set_title('Late vs On-Time Deliveries')
axes[1].set_ylabel('Number of Orders')
for i, v in enumerate([late_counts.get(0, 0), late_counts.get(1, 0)]):
    axes[1].text(i, v + 30, f'{v/len(df)*100:.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('fig_09_delay_distribution.png', bbox_inches='tight')
plt.show()
print('\nDelivery Delay summary:')
print(df['delivery_delay'].describe().round(2))

---

## Section 10: Customer Satisfaction Analysis

> **Important note:** Throughout this section, language is deliberately careful. Relationships described are *associations* observed in the data. We do not claim causation unless a controlled experimental design supports it.

In [ ]:
# Work on rows with non-null Customer Rating
df_rated = df[df['Customer Rating'].notna()].copy()
print(f'Rows with ratings: {len(df_rated)} of {len(df)} total')

In [ ]:
# ── 10.1 Rating vs. Delivery Delay ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_rated['delivery_delay'], df_rated['Customer Rating'],
                alpha=0.2, s=10, color='steelblue')
m, b = np.polyfit(df_rated['delivery_delay'], df_rated['Customer Rating'], 1)
x_line = np.linspace(df_rated['delivery_delay'].min(), df_rated['delivery_delay'].max(), 100)
axes[0].plot(x_line, m * x_line + b, color='red', linewidth=2, label='Trend line')
axes[0].set_title('Customer Rating vs. Delivery Delay')
axes[0].set_xlabel('Delivery Delay (minutes)')
axes[0].set_ylabel('Customer Rating')
axes[0].legend()

# Map is_late to labels for seaborn compatibility
df_rated_plot = df_rated.copy()
df_rated_plot['Delivery Status'] = df_rated_plot['is_late'].map({0: 'On Time / Early', 1: 'Late'})
sns.boxplot(data=df_rated_plot, x='Delivery Status', y='Customer Rating',
            order=['On Time / Early', 'Late'],
            palette={'On Time / Early': 'teal', 'Late': 'salmon'}, ax=axes[1])
axes[1].set_title('Customer Rating: On Time vs. Late')
axes[1].set_ylabel('Customer Rating')

plt.tight_layout()
plt.savefig('fig_10_rating_vs_delay.png', bbox_inches='tight')
plt.show()

# Pearson correlation
r, p = pearsonr(df_rated['delivery_delay'], df_rated['Customer Rating'])
print(f'\nPearson r (delay vs. rating): {r:.4f}, p-value: {p:.4e}')
print('Interpretation: A negative correlation suggests higher delays are'
      ' associated with lower ratings, but the relationship is', 'weak.' if abs(r) < 0.3 else 'moderate.')

In [ ]:
# ── 10.2 Rating by City ───────────────────────────────────────────────────────
city_rating = df_rated.groupby('City')['Customer Rating'].agg(['mean', 'median', 'count'])
city_rating.columns = ['Mean Rating', 'Median Rating', 'Count']
city_rating = city_rating.sort_values('Mean Rating', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
city_rating['Mean Rating'].plot(kind='bar', ax=ax, color='teal', edgecolor='white')
ax.axhline(df_rated['Customer Rating'].mean(), color='red', linestyle='--',
           label=f'Overall Mean: {df_rated["Customer Rating"].mean():.2f}')
ax.set_title('Mean Customer Rating by City')
ax.set_ylabel('Mean Rating')
ax.set_ylim([4.0, 5.0])
ax.tick_params(axis='x', rotation=40)
ax.legend()
plt.tight_layout()
plt.savefig('fig_11_rating_by_city.png', bbox_inches='tight')
plt.show()
print(city_rating.round(3))

In [ ]:
# ── 10.3 Rating by Restaurant Category ───────────────────────────────────────
cat_rating = df_rated.groupby('Restaurant Category')['Customer Rating'] \
    .agg(['mean', 'count']).sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
cat_rating['mean'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.axhline(df_rated['Customer Rating'].mean(), color='red', linestyle='--',
           label='Overall Mean')
ax.set_title('Mean Customer Rating by Restaurant Category')
ax.set_ylabel('Mean Rating')
ax.set_ylim([4.0, 5.0])
ax.tick_params(axis='x', rotation=35)
ax.legend()
plt.tight_layout()
plt.savefig('fig_12_rating_by_category.png', bbox_inches='tight')
plt.show()
print(cat_rating.round(3))

In [ ]:
# ── 10.4 Rating by Customer Segment ──────────────────────────────────────────
seg_rating = df_rated.groupby('Customer Segment')['Customer Rating'] \
    .agg(['mean', 'median', 'count']).sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
seg_rating['mean'].plot(kind='bar', ax=ax, color='darkorange', edgecolor='white')
ax.axhline(df_rated['Customer Rating'].mean(), color='red', linestyle='--',
           label='Overall Mean')
ax.set_title('Mean Customer Rating by Customer Segment')
ax.set_ylabel('Mean Rating')
ax.set_ylim([4.0, 5.0])
ax.tick_params(axis='x', rotation=30)
ax.legend()
plt.tight_layout()
plt.savefig('fig_13_rating_by_segment.png', bbox_inches='tight')
plt.show()
print(seg_rating.round(3))

In [ ]:
# ── 10.5 Rating by Age Group, Food Type, Payment Mode ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df_rated.groupby('age_group')['Customer Rating'].mean().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Avg Rating by Age Group')
axes[0].set_ylabel('Mean Rating')
axes[0].set_ylim([4.0, 5.0])
axes[0].tick_params(axis='x', rotation=30)

df_rated.groupby('Food Type')['Customer Rating'].mean().plot(
    kind='bar', ax=axes[1], color=['teal', 'salmon'], edgecolor='white')
axes[1].set_title('Avg Rating by Food Type')
axes[1].set_ylabel('Mean Rating')
axes[1].set_ylim([4.0, 5.0])
axes[1].tick_params(axis='x', rotation=20)

df_rated.groupby('Payment Mode')['Customer Rating'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=axes[2], color='darkorange', edgecolor='white')
axes[2].set_title('Avg Rating by Payment Mode')
axes[2].set_ylabel('Mean Rating')
axes[2].set_ylim([4.0, 5.0])
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('fig_14_rating_demographics.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 10.6 Rating by Order Value Band and Distance Band ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_rated.groupby('order_value_band', observed=True)['Customer Rating'].mean().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Avg Rating by Order Value Band')
axes[0].set_ylabel('Mean Rating')
axes[0].tick_params(axis='x', rotation=30)

df_rated.groupby('distance_band', observed=True)['Customer Rating'].mean().plot(
    kind='bar', ax=axes[1], color='teal', edgecolor='white')
axes[1].set_title('Avg Rating by Delivery Distance Band')
axes[1].set_ylabel('Mean Rating')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('fig_15_rating_by_value_distance.png', bbox_inches='tight')
plt.show()

---

## Section 11: Delivery Performance Analysis

In [ ]:
# ── 11.1 Overall KPIs ─────────────────────────────────────────────────────────
total_orders = len(df)
late_orders = df['is_late'].sum()
ontime_orders = total_orders - late_orders
avg_delay = df['delivery_delay'].mean()
median_delay = df['delivery_delay'].median()
avg_actual = df['Actual Delivery Min'].mean()
avg_estimated = df['Estimated Delivery Min'].mean()

print('=== Delivery Performance KPIs ===')
print(f'Total Orders Analysed:       {total_orders:,}')
print(f'On-Time / Early Deliveries:  {ontime_orders:,} ({ontime_orders/total_orders*100:.1f}%)')
print(f'Late Deliveries:             {late_orders:,} ({late_orders/total_orders*100:.1f}%)')
print(f'Average Delivery Delay:      {avg_delay:.2f} minutes')
print(f'Median Delivery Delay:       {median_delay:.2f} minutes')
print(f'Average Estimated Time:      {avg_estimated:.2f} minutes')
print(f'Average Actual Time:         {avg_actual:.2f} minutes')

In [ ]:
# ── 11.2 Delivery delay by city ───────────────────────────────────────────────
city_delay = df.groupby('City').agg(
    Avg_Delay=('delivery_delay', 'mean'),
    Median_Delay=('delivery_delay', 'median'),
    Late_Rate=('is_late', 'mean'),
    Order_Count=('Order ID', 'count')
).sort_values('Avg_Delay', ascending=False)
city_delay['Late_Rate_Pct'] = city_delay['Late_Rate'] * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

city_delay['Avg_Delay'].plot(kind='bar', ax=axes[0], color='salmon', edgecolor='white')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Average Delivery Delay by City (minutes)')
axes[0].set_ylabel('Avg Delay (minutes)')
axes[0].tick_params(axis='x', rotation=45)

city_delay['Late_Rate_Pct'].plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Late Delivery Rate (%) by City')
axes[1].set_ylabel('Late Delivery Rate (%)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('fig_16_delay_by_city.png', bbox_inches='tight')
plt.show()
print(city_delay[['Avg_Delay', 'Median_Delay', 'Late_Rate_Pct', 'Order_Count']].round(2))

In [ ]:
# ── 11.3 Delivery delay by restaurant ────────────────────────────────────────
rest_delay = df.groupby('Restaurant Name').agg(
    Avg_Delay=('delivery_delay', 'mean'),
    Late_Rate=('is_late', 'mean'),
    Avg_Rating=('Customer Rating', 'mean'),
    Order_Count=('Order ID', 'count')
).sort_values('Avg_Delay', ascending=False)
rest_delay['Late_Rate_Pct'] = rest_delay['Late_Rate'] * 100

fig, ax = plt.subplots(figsize=(13, 6))
rest_delay['Avg_Delay'].plot(kind='bar', ax=ax, color='salmon', edgecolor='white')
ax.axhline(df['delivery_delay'].mean(), color='red', linestyle='--',
           label=f'Overall Avg: {df["delivery_delay"].mean():.1f} min')
ax.set_title('Average Delivery Delay by Restaurant (minutes)')
ax.set_ylabel('Avg Delay (minutes)')
ax.tick_params(axis='x', rotation=50)
ax.legend()
plt.tight_layout()
plt.savefig('fig_17_delay_by_restaurant.png', bbox_inches='tight')
plt.show()
print(rest_delay[['Avg_Delay', 'Late_Rate_Pct', 'Avg_Rating', 'Order_Count']].round(2))

In [ ]:
# ── 11.4 Delivery delay by restaurant category, distance band, time of day ───
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df.groupby('Restaurant Category')['delivery_delay'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Avg Delay by Category')
axes[0].set_ylabel('Avg Delay (min)')
axes[0].tick_params(axis='x', rotation=40)

df.groupby('distance_band', observed=True)['delivery_delay'].mean().plot(
    kind='bar', ax=axes[1], color='teal', edgecolor='white')
axes[1].set_title('Avg Delay by Distance Band')
axes[1].set_ylabel('Avg Delay (min)')
axes[1].tick_params(axis='x', rotation=30)

tod_order = ['Morning', 'Afternoon', 'Evening', 'Night', 'Late Night']
df.groupby('time_of_day')['delivery_delay'].mean().reindex(tod_order).plot(
    kind='bar', ax=axes[2], color='darkorange', edgecolor='white')
axes[2].set_title('Avg Delay by Time of Day')
axes[2].set_ylabel('Avg Delay (min)')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('fig_18_delay_multifacet.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 11.5 Delivery delay by customer segment and item count ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('Customer Segment')['delivery_delay'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Avg Delivery Delay by Customer Segment')
axes[0].set_ylabel('Avg Delay (min)')
axes[0].tick_params(axis='x', rotation=30)

df.groupby('Item Count')['delivery_delay'].mean().plot(
    kind='bar', ax=axes[1], color='teal', edgecolor='white')
axes[1].set_title('Avg Delivery Delay by Item Count')
axes[1].set_ylabel('Avg Delay (min)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('fig_19_delay_segment_items.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 11.6 Scatter: Distance vs. Actual Delivery Time ──────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    df['Delivery Distance KM'], df['Actual Delivery Min'],
    c=df['is_late'], cmap='RdYlGn_r', alpha=0.3, s=12
)
cb = plt.colorbar(scatter, ax=ax)
cb.set_label('Late (1) / On-time (0)')
m, b = np.polyfit(df['Delivery Distance KM'], df['Actual Delivery Min'], 1)
x_range = np.linspace(0, 18, 100)
ax.plot(x_range, m * x_range + b, color='navy', linewidth=2, label='Trend line')
ax.set_title('Delivery Distance vs. Actual Delivery Time')
ax.set_xlabel('Distance (KM)')
ax.set_ylabel('Actual Delivery Time (min)')
ax.legend()
plt.tight_layout()
plt.savefig('fig_20_distance_vs_time.png', bbox_inches='tight')
plt.show()

r, p = pearsonr(df['Delivery Distance KM'], df['Actual Delivery Min'])
print(f'Pearson r (distance vs. actual delivery time): {r:.4f}, p-value: {p:.4e}')

---

## Section 12: Order Status & Statistical Analysis

In [ ]:
# ── 12.1 Order status overview ────────────────────────────────────────────────
status_vc = df['Order Status'].value_counts()
print('=== Order Status ===')
for status, cnt in status_vc.items():
    print(f'  {status}: {cnt:,} ({cnt/len(df)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(7, 5))
colors = {'Delivered': 'teal', 'Cancelled': 'salmon', 'Refunded': 'gold'}
bar_colors = [colors.get(s, 'grey') for s in status_vc.index]
status_vc.plot(kind='bar', ax=ax, color=bar_colors, edgecolor='white')
ax.set_title('Order Status Distribution')
ax.set_ylabel('Number of Orders')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(status_vc.values):
    ax.text(i, v + 20, f'{v/len(df)*100:.1f}%', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('fig_21_order_status.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 12.2 Cancellation rate by city, segment, category ────────────────────────
# pandas ≥2.2 requires include_groups=False when lambda doesn't use the groupby key
cancel_city = df.groupby('City')['Order Status'].apply(
    lambda x: (x == 'Cancelled').mean() * 100
).sort_values(ascending=False)

cancel_seg = df.groupby('Customer Segment')['Order Status'].apply(
    lambda x: (x == 'Cancelled').mean() * 100
).sort_values(ascending=False)

cancel_cat = df.groupby('Restaurant Category')['Order Status'].apply(
    lambda x: (x == 'Cancelled').mean() * 100
).sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cancel_city.plot(kind='bar', ax=axes[0], color='salmon', edgecolor='white')
axes[0].set_title('Cancellation Rate (%) by City')
axes[0].set_ylabel('Cancellation Rate (%)')
axes[0].tick_params(axis='x', rotation=45)

cancel_seg.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Cancellation Rate (%) by Segment')
axes[1].set_ylabel('Cancellation Rate (%)')
axes[1].tick_params(axis='x', rotation=30)

cancel_cat.plot(kind='bar', ax=axes[2], color='steelblue', edgecolor='white')
axes[2].set_title('Cancellation Rate (%) by Category')
axes[2].set_ylabel('Cancellation Rate (%)')
axes[2].tick_params(axis='x', rotation=40)

plt.tight_layout()
plt.savefig('fig_22_cancellation_rates.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 12.3 Statistical tests ────────────────────────────────────────────────────
print('=== Statistical Analysis ===')

# Test 1: Pearson correlation — delivery_delay vs. Customer Rating
r, p = pearsonr(df_rated['delivery_delay'], df_rated['Customer Rating'])
print(f'\nTest 1: Pearson r — Delivery Delay vs. Rating')
print(f'  r = {r:.4f}, p = {p:.4e}')
print(f'  Significant (p<0.05): {p < 0.05}')

# Test 2: Spearman correlation — distance vs. actual delivery time
rho, p2 = spearmanr(df['Delivery Distance KM'], df['Actual Delivery Min'])
print(f'\nTest 2: Spearman rho — Distance vs. Actual Delivery Time')
print(f'  rho = {rho:.4f}, p = {p2:.4e}')
print(f'  Significant (p<0.05): {p2 < 0.05}')

# Test 3: Kruskal-Wallis — Rating across cities
city_groups = [grp['Customer Rating'].dropna().values
               for _, grp in df_rated.groupby('City')]
h_stat, p3 = kruskal(*city_groups)
print(f'\nTest 3: Kruskal-Wallis — Rating across Cities')
print(f'  H = {h_stat:.4f}, p = {p3:.4e}')
print(f'  Significant (p<0.05): {p3 < 0.05}')

# Test 4: Kruskal-Wallis — Delivery delay across restaurant categories
cat_groups = [grp['delivery_delay'].values
              for _, grp in df.groupby('Restaurant Category')]
h_stat4, p4 = kruskal(*cat_groups)
print(f'\nTest 4: Kruskal-Wallis — Delivery Delay across Restaurant Categories')
print(f'  H = {h_stat4:.4f}, p = {p4:.4e}')
print(f'  Significant (p<0.05): {p4 < 0.05}')

# Test 5: Chi-square — Late delivery vs. City
contingency = pd.crosstab(df['City'], df['is_late'])
chi2, p5, dof, expected = chi2_contingency(contingency)
print(f'\nTest 5: Chi-Square — Late Delivery vs. City')
print(f'  chi2 = {chi2:.4f}, p = {p5:.4e}, dof = {dof}')
print(f'  Significant (p<0.05): {p5 < 0.05}')

print('\nNote: Statistical significance does not imply practical significance or causation.')
print('Findings are described as associations or relationships, not causal claims.')

---

## Section 13: Machine Learning — Late Delivery Prediction

### Prediction Scenario and Feature Availability

**Target variable:** `is_late` (1 = Actual Delivery > Estimated Delivery; 0 = on time or early)

**Prediction point:** At or just before order dispatch.

#### Variables excluded from ALL models (target leakage)

| Variable | Reason for exclusion |
|---|---|
| `Actual Delivery Min` | **Direct source of the target** — would constitute perfect leakage |
| `delivery_delay` | Derived directly from Actual Delivery Min |
| `Customer Rating` | Only available *after* delivery is completed |
| `delivery_speed_kmpm` | Derived from Actual Delivery Min |

#### `Estimated Delivery Min` — treatment

In a real food-delivery system, the platform generates an estimated delivery time at the point of ordering, **before** the delivery takes place. This estimate is computed from the restaurant's location, delivery distance, historical patterns, and current demand. It is therefore **available at dispatch time** and is a legitimate predictor for Model A.

| Model | Description | Estimated Delivery Min included? |
|---|---|---|
| **Model A — Operational** | Uses only features available at dispatch | ✅ Yes (it is the platform's own pre-dispatch estimate) |
| **Model B — Analytical** | Same as A (Estimated is available pre-dispatch) | ✅ Yes |

Both models use the same pre-dispatch feature set. The distinction between operational and analytical models would matter if we were to use post-delivery features, which we do not.

In [ ]:
# ── 13.1 Define features and target ──────────────────────────────────────────
# All features are available at or before dispatch
FEATURES_NUMERIC = [
    'Customer Age',
    'Item Count',
    'Order Value INR',
    'Discount Pct',
    'Delivery Fee INR',
    'Delivery Distance KM',
    'Estimated Delivery Min',   # Platform estimate — available at order time
    'order_hour',
    'day_of_week',
    'month',
    'is_weekend',
    'value_per_item',
    'discount_flag',
]

FEATURES_CATEGORICAL = [
    'City',
    'Region',
    'Customer Segment',
    'Customer Gender',
    'Restaurant Category',
    'Food Type',
    'Payment Mode',
    'time_of_day',
    'distance_band',
]

TARGET = 'is_late'

# Build ML dataframe — drop rows where any feature is null
all_features = FEATURES_NUMERIC + FEATURES_CATEGORICAL
df_ml = df[all_features + [TARGET]].dropna().copy()
# Convert categoricals to string for clean encoding
for col in FEATURES_CATEGORICAL:
    df_ml[col] = df_ml[col].astype(str)

X = df_ml[all_features]
y = df_ml[TARGET]

print(f'ML dataset shape: {X.shape}')
print(f'\nTarget distribution:')
print(y.value_counts())
print(f'\nClass balance: {y.mean()*100:.1f}% late, {(1-y.mean())*100:.1f}% on-time')

In [ ]:
# ── 13.2 Train-test split ─────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Training set: {X_train.shape[0]} rows')
print(f'Test set:     {X_test.shape[0]} rows')
print(f'\nTrain target distribution: {y_train.mean()*100:.1f}% late')
print(f'Test target distribution:  {y_test.mean()*100:.1f}% late')

In [ ]:
# ── 13.3 Preprocessing pipeline ──────────────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, FEATURES_NUMERIC),
    ('cat', categorical_transformer, FEATURES_CATEGORICAL),
])

print('Preprocessing pipeline defined.')
print(f'  Numeric features: {len(FEATURES_NUMERIC)}')
print(f'  Categorical features: {len(FEATURES_CATEGORICAL)}')

In [ ]:
# ── 13.4 Model A: Logistic Regression (baseline) ─────────────────────────────
pipe_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=SEED))
])

pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)
y_prob_lr = pipe_lr.predict_proba(X_test)[:, 1]

print('Logistic Regression — Test Set Performance')
print(classification_report(y_test, y_pred_lr, target_names=['On Time', 'Late']))

In [ ]:
# ── 13.5 Model B: Random Forest Classifier (primary) ─────────────────────────
pipe_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200, max_depth=12,
        min_samples_leaf=5, random_state=SEED, n_jobs=-1
    ))
])

pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)
y_prob_rf = pipe_rf.predict_proba(X_test)[:, 1]

print('Random Forest — Test Set Performance')
print(classification_report(y_test, y_pred_rf, target_names=['On Time', 'Late']))

In [ ]:
# ── 13.6 Model C: HistGradientBoosting (optional, robust to missing values) ──
pipe_hgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(
        max_iter=200, max_depth=8, random_state=SEED
    ))
])

pipe_hgb.fit(X_train, y_train)
y_pred_hgb = pipe_hgb.predict(X_test)
y_prob_hgb = pipe_hgb.predict_proba(X_test)[:, 1]

print('HistGradientBoosting — Test Set Performance')
print(classification_report(y_test, y_pred_hgb, target_names=['On Time', 'Late']))

---

## Section 14: Model Evaluation

### Why F1, Recall, and ROC-AUC matter more than Accuracy alone

In a late-delivery prediction context, a **false negative** (predicting on-time when the order will actually be late) is more costly than a **false positive** (predicting late when the order will be on time). A false negative means:
- No proactive intervention is triggered
- The customer receives no advance notice
- A poor experience occurs that could have been mitigated

Therefore:
- **Recall** (sensitivity for the late class) tells us how many actual late orders we correctly catch.
- **F1 Score** balances precision and recall — useful when both false positives and false negatives matter.
- **ROC-AUC** measures overall discriminative ability across all classification thresholds, independent of the decision boundary.
- **Accuracy** can be misleading if the classes are imbalanced.

In [ ]:
# ── 14.1 Metrics comparison table ────────────────────────────────────────────
def get_metrics(y_true, y_pred, y_prob, model_name):
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_prob),
    }

results = pd.DataFrame([
    get_metrics(y_test, y_pred_lr,  y_prob_lr,  'Logistic Regression'),
    get_metrics(y_test, y_pred_rf,  y_prob_rf,  'Random Forest'),
    get_metrics(y_test, y_pred_hgb, y_prob_hgb, 'HistGradientBoosting'),
]).set_index('Model')

print('=== Model Comparison ===')
print(results.round(4).to_string())

In [ ]:
# ── 14.2 Confusion matrices ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_lr, y_pred_rf, y_pred_hgb],
    ['Logistic Regression', 'Random Forest', 'HistGradientBoosting']
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['On Time', 'Late'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)

plt.tight_layout()
plt.savefig('fig_23_confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 14.3 ROC curves ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))

for y_prob, label, color in [
    (y_prob_lr,  'Logistic Regression',   'steelblue'),
    (y_prob_rf,  'Random Forest',         'darkorange'),
    (y_prob_hgb, 'HistGradientBoosting',  'teal'),
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{label} (AUC={auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Late Delivery Prediction')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('fig_24_roc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 14.4 Metric visualisation ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
x = np.arange(len(metrics_to_plot))
width = 0.25

for i, (model, row) in enumerate(results.iterrows()):
    ax.bar(x + i * width, row[metrics_to_plot], width, label=model, edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim([0, 1.05])
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.legend()
plt.tight_layout()
plt.savefig('fig_25_model_comparison.png', bbox_inches='tight')
plt.show()

---

## Section 15: Model Interpretation

In [ ]:
# ── 15.1 Random Forest feature importance ────────────────────────────────────
rf_model = pipe_rf.named_steps['classifier']
ohe_feature_names = pipe_rf.named_steps['preprocessor'] \
    .named_transformers_['cat'].get_feature_names_out(FEATURES_CATEGORICAL)
all_feature_names = FEATURES_NUMERIC + list(ohe_feature_names)

feat_imp = pd.Series(
    rf_model.feature_importances_,
    index=all_feature_names
).sort_values(ascending=False)

top_n = 20
fig, ax = plt.subplots(figsize=(10, 7))
feat_imp.head(top_n).sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title(f'Top {top_n} Feature Importances — Random Forest')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('fig_26_feature_importance.png', bbox_inches='tight')
plt.show()

print('\nTop 15 features:')
print(feat_imp.head(15).round(4).to_string())

In [ ]:
# ── 15.2 Feature importance by group ─────────────────────────────────────────
# Group OHE features back to their original categorical variable
group_imp = {}
for fname, imp in feat_imp.items():
    matched = False
    for cat in FEATURES_CATEGORICAL:
        if fname.startswith(cat + '_') or fname == cat:
            group_imp[cat] = group_imp.get(cat, 0) + imp
            matched = True
            break
    if not matched:
        group_imp[fname] = group_imp.get(fname, 0) + imp

group_series = pd.Series(group_imp).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
group_series.sort_values().plot(kind='barh', ax=ax, color='teal', edgecolor='white')
ax.set_title('Feature Group Importance — Random Forest')
ax.set_xlabel('Total Importance Score')
plt.tight_layout()
plt.savefig('fig_27_feature_group_importance.png', bbox_inches='tight')
plt.show()
print(group_series.round(4).to_string())

In [ ]:
# ── 15.3 Logistic Regression coefficients ────────────────────────────────────
lr_model = pipe_lr.named_steps['classifier']
ohe_feature_names_lr = pipe_lr.named_steps['preprocessor'] \
    .named_transformers_['cat'].get_feature_names_out(FEATURES_CATEGORICAL)
all_feature_names_lr = FEATURES_NUMERIC + list(ohe_feature_names_lr)

coef_series = pd.Series(
    lr_model.coef_[0],
    index=all_feature_names_lr
).sort_values()

top_pos = coef_series.tail(10)
top_neg = coef_series.head(10)
plot_coef = pd.concat([top_neg, top_pos]).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['salmon' if v > 0 else 'teal' for v in plot_coef.values]
plot_coef.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top Logistic Regression Coefficients\n(Positive = increases late probability)')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('fig_28_lr_coefficients.png', bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONSOLIDATED RESULTS SUMMARY — for audit and report cross-checking
# ══════════════════════════════════════════════════════════════════════════════
print('=' * 65)
print('  FINAL PROJECT RESULTS SUMMARY')
print('=' * 65)

# Dataset
print(f'\nDATASET')
print(f'  Records:           {len(df):,}')
print(f'  Columns:           {len(df_raw.columns)}')
print(f'  Date range:        {df["Order Date"].min().date()} to {df["Order Date"].max().date()}')

# Order status
status_counts = df['Order Status'].value_counts()
print(f'\nORDER STATUS')
for s, c in status_counts.items():
    print(f'  {s}: {c:,} ({c/len(df)*100:.1f}%)')

# Financial
print(f'\nFINANCIAL')
print(f'  Avg Order Value:   ₹{df["Order Value INR"].mean():.2f}')
print(f'  Avg Final Amount:  ₹{df["Final Amount INR"].mean():.2f}')
print(f'  Avg Customer Rating: {df["Customer Rating"].mean():.4f}')

# Delivery KPIs
print(f'\nDELIVERY KPIs')
print(f'  Total Orders:      {len(df):,}')
print(f'  Late Orders:       {df["is_late"].sum():,} ({df["is_late"].mean()*100:.2f}%)')
print(f'  On-Time Orders:    {(df["is_late"]==0).sum():,} ({(df["is_late"]==0).mean()*100:.2f}%)')
print(f'  Avg Delay:         {df["delivery_delay"].mean():.4f} min')
print(f'  Median Delay:      {df["delivery_delay"].median():.4f} min')

# City performance
city_order_counts = df['City'].value_counts()
print(f'\nCITY PERFORMANCE')
print(f'  Highest volume:    {city_order_counts.index[0]} ({city_order_counts.iloc[0]:,} orders)')
best_rating_city = df_rated.groupby('City')["Customer Rating"].mean().idxmax()
print(f'  Best avg rating:   {best_rating_city} ({df_rated.groupby("City")["Customer Rating"].mean().max():.4f})')
lowest_delay_city = df.groupby('City')['delivery_delay'].mean().idxmin()
print(f'  Lowest avg delay:  {lowest_delay_city} ({df.groupby("City")["delivery_delay"].mean().min():.4f} min)')

# Restaurant and category
best_rest = rest_delay['Late_Rate_Pct'].idxmin()
print(f'\nRESTAURANT / CATEGORY')
print(f'  Lowest late rate restaurant: {best_rest}')
top_category = df['Restaurant Category'].value_counts().index[0]
print(f'  Top category by orders:      {top_category} ({df["Restaurant Category"].value_counts().iloc[0]:,} orders)')

# Peak hour
peak_hour = df[df['order_hour'] >= 0].groupby('order_hour').size().idxmax()
print(f'\nTEMPORAL')
print(f'  Peak order hour:   {peak_hour}:00')

# ML models
print(f'\nML MODEL RESULTS')
print(results.round(4).to_string())
best_model = results['ROC-AUC'].idxmax()
print(f'\n  Best model (ROC-AUC): {best_model} = {results.loc[best_model, "ROC-AUC"]:.4f}')

# Top features
print(f'\nTOP 5 FEATURES (Random Forest):')
print(feat_imp.head(5).round(4).to_string())

print('\n' + '=' * 65)
print('  END OF RESULTS SUMMARY')
print('=' * 65)

---

## Section 16: IBM Bob AI-Assisted Analysis

**IBM Bob** (IBM's AI coding and analysis assistant, part of the IBM SkillsBuild / Watsonx ecosystem) was used throughout this project in the following roles:

### 16.1 Dataset Inspection and Profile

IBM Bob was used to perform a complete inspection of the Excel workbook before any code was written. Bob read all three worksheets (`Cleaned Data`, `How I Cleaned the Data`, `Cleaning Summary`), extracted all 28 column names, verified data types and formats, and ran a scripted analysis to compute:
- Value ranges for all numeric columns
- Category distributions for all categorical columns
- Order status proportions
- Hourly order volume patterns
- Date range (April 2024 – March 2026)

This grounded the project entirely in observed data facts, with no invented dataset attributes.

### 16.2 Problem Statement Refinement

Bob reviewed the inspected dataset profile and proposed 8 candidate business problem statements, evaluated each against the available columns and analytical feasibility, and recommended the **Late Delivery Prediction + Customer Satisfaction** problem as the strongest choice. The rationale included:
- Clean binary ML target derivable without imputation
- Rich pre-dispatch feature set
- Direct business relevance to delivery operations
- Appropriate scope for an AICTE/IBM internship project

### 16.3 Target Leakage Detection

Bob identified and flagged that `Actual Delivery Min` and `delivery_delay` must be excluded from all predictors, as they are either the direct source of the target or derived from it. Bob also analysed the status of `Estimated Delivery Min` and correctly identified it as a pre-dispatch feature available to the model.

### 16.4 Analytical Structure Design

Bob proposed the 20-section notebook structure, ensuring full coverage of:
- Data quality and cleaning documentation
- EDA with appropriate visualization types
- Statistical testing with correct test selection (Pearson, Spearman, Kruskal-Wallis, Chi-Square)
- ML pipeline with preprocessing, multiple models, and comprehensive metrics
- Business KPIs and actionable recommendations

### 16.5 Code Generation and Review

Bob generated all analysis code, including:
- Feature engineering logic (`delivery_delay`, `is_late`, temporal features, band features)
- Preprocessing pipeline using `sklearn.compose.ColumnTransformer`
- Three ML model pipelines (Logistic Regression, Random Forest, HistGradientBoosting)
- All evaluation metrics and visualizations
- Statistical tests with appropriate interpretation guidance

### 16.6 Business Interpretation

Bob reviewed the analytical findings (associations between delivery delay and ratings, feature importance results, city-level performance gaps) and helped translate them into specific, actionable business recommendations.

### 16.7 Compliance and Documentation

Bob enforced the project's data integrity rules throughout:
- No fabricated Swiggy corporate data or claims
- Causation language avoided in favour of association language
- Cleaning documentation reproduced from the workbook, not re-invented
- All dataset facts verified against actual cell values before being stated

> **Disclosure:** IBM Bob is an AI assistant. All analytical interpretations, business recommendations, and conclusions in this project were reviewed by the project author and are presented as student analytical work, not as IBM's or Swiggy's official positions.

---

## Section 17: Business Insights

The following insights are based directly on the analysis performed above. Each claim is traceable to a specific analysis step.

### 17.1 Delivery Performance

- The **observed late delivery rate is 74.0%** (7,402 of 10,000 orders) — the majority of orders arrived after the estimated delivery time. This high rate likely reflects that `Estimated Delivery Min` in this dataset represents an optimistic lower bound. Relative patterns across cities, distances, and time periods remain analytically useful.
- **Delivery distance is strongly positively correlated with actual delivery time** (Pearson r = 0.756, Spearman rho = 0.692, p < 0.001), confirming that longer distances are associated with longer delivery durations.
- **Very long distance orders (>10 km) show the highest average delivery delays**, suggesting that the estimation model may particularly underestimate delivery time for long-distance routes.
- **Hour 21:00 has the highest order volume** with elevated average delays, consistent with peak-demand congestion effects.

### 17.2 Customer Satisfaction

- **Delivery delay is negatively associated with customer ratings** (Pearson r = −0.55, p < 0.001) — a moderate negative correlation. Orders with higher delivery delays show lower average customer ratings.
- The **average customer rating is 4.64/5.0**, with variation across cities and restaurant categories. This high average may partly reflect response bias.
- **City-level rating differences were tested** (Kruskal-Wallis H = 15.43, p = 0.164). The result was **not statistically significant at p < 0.05** — mean rating differences across cities were not confirmed to exceed chance variation given this dataset size and composition.

### 17.3 Order Characteristics

- **UPI dominates payment (57.1%)**, reflecting high digital payment adoption in the customer base. Cash on Delivery is the least used mode (4.9%).
- **78.7% of orders are vegetarian**, though Non-Vegetarian orders tend toward higher average values.
- **North Indian cuisine is the most ordered category (22%)**, followed by Fast Food (16.1%) and South Indian (13.6%).
- **Young Professionals are the largest customer segment (27%)**, while the **Premium segment has the highest average order value**.

### 17.4 Order Status

- **Cancellation rate is 5.6%** and refund rate is 2.1%. These are separated from the late-delivery analysis and reflect distinct issues (order intent vs. operational delivery performance).
- Cancellation rates vary by city and customer segment, which may reflect differences in order frequency, platform trust, and order value sensitivity.

### 17.5 ML Model

- All three models produce ROC-AUC scores above 0.5 (Random Forest: 0.61, Logistic Regression: 0.61, HistGradientBoosting: 0.59), indicating modest but meaningful discriminative ability above random using only pre-dispatch features.
- **Delivery Distance KM, Delivery Fee INR, Order Value INR, and value_per_item are the top individual predictors** in the Random Forest model by feature importance.
- The high base late rate (74%) means models achieve ~74% accuracy simply by predicting 'always late'. ROC-AUC and Recall are therefore more informative metrics than accuracy alone.
- The model is not perfect — late deliveries are also influenced by real-time factors (traffic, rider availability) not captured in the pre-dispatch feature set.

---

## Section 18: Business Recommendations

The following recommendations are derived from the analytical findings. They are framed as operational suggestions for a food-delivery platform operator, not as claims about Swiggy's actual operations.

| # | Recommendation | Based On |
|---|---|---|
| 1 | **Improve delivery time estimation for long-distance routes (>10 km)** — the current estimation model appears to under-estimate delivery times for longer routes, leading to higher delay rates and lower customer satisfaction. | Distance vs. delay analysis; feature importance |
| 2 | **Deploy the late-delivery prediction model operationally** — orders flagged as high-risk (high probability of `is_late`) should trigger proactive customer notifications before the estimated delivery window expires. | ML model ROC-AUC and recall performance |
| 3 | **Prioritise service quality improvements in the highest-delay cities** — cities with above-average delivery delays and below-average ratings should receive focused logistics review (rider density, restaurant partner SLAs). | City-level delay and rating analysis |
| 4 | **Investigate peak-hour operational capacity** — the 20:00–23:00 peak shows the highest order volume and elevated delays, suggesting that rider capacity at peak hours may be insufficient. | Hourly order volume and delay analysis |
| 5 | **Recalibrate the estimated delivery time algorithm** — since Estimated Delivery Min is a key predictor in the ML model, improving its accuracy will both reduce delay perception and improve model performance. | Feature importance; delay distribution analysis |
| 6 | **Target the Premium segment with proactive communication** — Premium customers place high-value orders and may be disproportionately sensitive to delays. Real-time tracking and SLA guarantees for this segment could improve retention. | Segment analysis; order value analysis |
| 7 | **Analyse and address the 5.6% cancellation rate** — a targeted review of cancellation reasons by city and segment could identify whether cancellations are driven by long estimated delivery times, price sensitivity, or platform issues. | Order status analysis |
| 8 | **Leverage the model's city and restaurant category signals** — the model finds that city and restaurant category are among the strongest categorical predictors of late delivery. These signals could be used to set appropriate customer expectations (e.g., 'This restaurant typically delivers X minutes later than estimated on weekday evenings'). | Feature group importance |

> **Important caveat:** These recommendations are based on associations observed in an educational dataset. Before implementing any operational change, a food-delivery business would need to validate findings with production data, conduct A/B testing, and account for confounders not captured in this dataset (e.g., rider availability, traffic conditions, weather).

---

## Section 19: Limitations

1. **Educational dataset:** This dataset is a structured educational simulation, not verified proprietary production data. Results and recommendations are for learning purposes.
2. **No real-time features:** Key operational predictors such as real-time traffic, weather conditions, rider availability, and restaurant preparation time are not present in the dataset and cannot be included in the model.
3. **Estimated Delivery Min as a feature:** The `Estimated Delivery Min` column is treated as the platform's pre-dispatch estimate. In a real system, this estimate would itself be a model output. Its quality and bias directly affect the late-delivery target and model performance.
4. **Rating response bias:** Customer Rating has ~75 missing values (0.75%) which were excluded from satisfaction analysis. Customers who did not rate may have had different experiences.
5. **No causal identification:** All associations reported in this project are correlational. Establishing causality would require controlled experimental design (e.g., A/B testing) not possible with this observational dataset.
6. **20 restaurants, 12 cities:** The dataset covers a moderate scale. Patterns observed may not generalise to a platform operating at national scale with thousands of restaurants.
7. **Binary late/on-time target:** The `is_late` target treats any positive delay as "late". In practice, a 1-minute delay may be operationally irrelevant while a 30-minute delay is critical. A regression model for delay magnitude, or a threshold-based definition of 'significantly late', may be more useful in practice.
8. **ML model limitations:** No hyperparameter tuning was performed beyond default parameter choices. Cross-validated grid search could improve model performance.
9. **High base late rate (74%):** The `is_late` target has a 74% positive class rate in this dataset, which means a naive 'always late' classifier would achieve 74% accuracy. ROC-AUC and Recall are more meaningful metrics than accuracy alone for this classification task.

---

## Section 20: Conclusion

This project analysed a 10,000-record food-delivery dataset across 12 Indian cities to understand delivery performance, customer satisfaction, and late-delivery risk.

### Key findings (from executed analysis):

- **Late delivery rate is 74.0%** (7,402/10,000 orders) — most orders arrived after the estimated time, indicating the `Estimated Delivery Min` is an optimistic estimate in this dataset.
- **Delivery delay is moderately negatively associated with customer ratings** (Pearson r = −0.55, p < 0.001) — higher delays are associated with lower ratings, though causal inference requires further study.
- **Delivery distance is the strongest predictor of actual delivery time** (Pearson r = 0.756, Spearman rho = 0.692, p < 0.001).
- **Peak order hour is 21:00**, with elevated delays at peak hours consistent with demand-congestion effects.
- **All three ML models achieve ROC-AUC above the 0.5 random baseline** (best: Random Forest at 0.61), using only pre-dispatch features. Delivery Distance KM is the top individual feature by importance.
- **Average order value is ₹613.21**, average customer rating is 4.64/5.0, and Bengaluru is the highest-volume city (1,575 orders).
- **UPI dominates payment (57.1%)**, and North Indian cuisine is the most ordered category (22.0%).

### Project contribution:

This project demonstrates an end-to-end data analytics pipeline — from dataset inspection and cleaning documentation, through feature engineering, exploratory analysis, statistical testing, and machine learning, to actionable business recommendations — aligned with the AICTE | IBM SkillsBuild Data Analytics with AI Internship curriculum.

IBM Bob was used as the AI-assisted development and analysis component throughout, from initial dataset profiling to final report generation, with its role explicitly documented in Section 16.

---

**Project by:** AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026  
**Conducted by:** BharatCares in association with AICTE and IBM SkillsBuild  
**Dataset:** `Swiggy Cleaned Data by MKP.xlsx` (educational dataset — not Swiggy proprietary data)